
# AEGIS-RS Fog Module: FFA-Net Training on RTTS

This notebook builds a research-oriented fog pipeline around **FFA-Net** for dehazing, then reports:
- Dehazing metrics: PSNR, SSIM, MSE
- Classification metrics: Accuracy, Precision, Recall, F1
- Optional detection metrics: IoU, AP, mAP (if YOLO predictions are provided)
- System metrics: FPS and latency

## Core Formulas Used

1. Atmospheric model: $I(x)=J(x)t(x)+A(1-t(x))$
2. Transmission: $t(x)=e^{-\beta d(x)}$
3. Reconstruction: $J(x)=\frac{I(x)-A}{t(x)}+A$
4. Convolution: $Z=W*X+b$
5. ReLU: $f(x)=\max(0,x)$
6. IoU: $IoU=\frac{|B\cap G|}{|B\cup G|}$
7. Sigmoid: $P=\frac{1}{1+e^{-x}}$
8. Cross-entropy: $L=-\sum y\log(\hat{y})$
9. PSNR: $PSNR=10\log_{10}(MAX^2/MSE)$
10. Risk score: $R=w_1F+w_2H+w_3T+w_4V^{-1}$

In [1]:
# If needed, uncomment and run once (CUDA / RTX 3070):
# !python -m pip install --upgrade pip
# !python -m pip install -r requirements-cuda.txt
# If you want CPU later:
# !python -m pip install -r requirements-cpu.txt

import json
import math
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print('Torch version:', torch.__version__)
print('Torch CUDA build:', torch.version.cuda)

# Important: if this shows a CPU build, restart the notebook kernel and run this cell again.
if '+cpu' in torch.__version__ or torch.version.cuda is None:
    raise RuntimeError(
        'This kernel is using a CPU-only torch build in memory. '
        'Restart the notebook kernel, ensure requirements-cuda.txt is installed in this same venv, '
        'then rerun from the first code cell.'
    )

torch.backends.cudnn.benchmark = True
if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is still unavailable to this kernel. Check NVIDIA driver/CUDA runtime, then restart kernel.'
    )

DEVICE = torch.device('cuda')
print('Device:', DEVICE)
print('GPU:', torch.cuda.get_device_name(0))

e:\6th SEM Data\Projects\AEGIS-RS_IDP\RTTS\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch version: 2.5.1+cu121
Torch CUDA build: 12.1
Device: cuda
GPU: NVIDIA GeForce RTX 3070 Laptop GPU


In [2]:
# Paths aligned to your workspace
RTTS_ROOT = Path('..').resolve()
JPEG_DIR = RTTS_ROOT / 'JPEGImages'
ANN_DIR = RTTS_ROOT / 'Annotations'
SPLIT_FILE = RTTS_ROOT / 'ImageSets' / 'Main' / 'test.txt'

MODEL_DIR = Path('models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 15  # Increased from 8
LR = 2e-4    # Slightly increased initial LR
VAL_RATIO = 0.15

print('RTTS root:', RTTS_ROOT)
print('Split file:', SPLIT_FILE)

RTTS root: E:\6th SEM Data\Projects\AEGIS-RS_IDP\RTTS
Split file: E:\6th SEM Data\Projects\AEGIS-RS_IDP\RTTS\ImageSets\Main\test.txt


In [3]:
def load_ids(split_file: Path) -> List[str]:
    ids = []
    with split_file.open('r', encoding='utf-8') as f:
        for line in f:
            v = line.strip()
            if v:
                ids.append(v)
    return ids

def read_rgb(path: Path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f'Image not found: {path}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def pseudo_clean_from_fog(rgb: np.ndarray) -> np.ndarray:
    # Improved target: CLAHE + gamma + slight contrast boost
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    l2 = clahe.apply(l)
    out = cv2.cvtColor(cv2.merge([l2, a, b]), cv2.COLOR_LAB2RGB).astype(np.float32) / 255.0
    gamma = 1.15
    out = np.power(np.clip(out, 1e-6, 1.0), 1.0 / gamma)
    # Slight contrast boost
    out = (out - 0.5) * 1.1 + 0.5
    return np.clip(out, 0.0, 1.0)

def synthesize_fog(clean: np.ndarray, beta_range=(0.4, 1.8), depth_variation=True) -> Tuple[np.ndarray, float]:
    h, w, _ = clean.shape
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    
    if depth_variation:
        # More realistic depth: Gaussian center with variation
        cy, cx = h * 0.4 + np.random.uniform(-50, 50), w * 0.5 + np.random.uniform(-50, 50)
        dy = (yy - cy) / max(h, 1)
        dx = (xx - cx) / max(w, 1)
        depth = np.exp(-(dy**2 + dx**2) / 0.3) * 0.6 + 0.4
    else:
        depth = (1.0 - yy / max(h - 1, 1))
        depth = 0.2 + 0.8 * depth

    beta = float(np.random.uniform(*beta_range))
    t = np.exp(-beta * depth)[..., None]
    
    # More varied airlight
    A = np.random.uniform(0.65, 0.95, size=(1, 1, 3)).astype(np.float32)
    foggy = clean * t + A * (1.0 - t)
    return np.clip(foggy, 0.0, 1.0), beta

def beta_to_fog_class(beta: float) -> int:
    # Better class separation
    if beta < 0.75:
        return 0
    if beta < 1.25:
        return 1
    return 2

ids = load_ids(SPLIT_FILE)
paths = [JPEG_DIR / f'{x}.png' for x in ids if (JPEG_DIR / f'{x}.png').exists()]
random.shuffle(paths)
split_idx = int(len(paths) * (1 - VAL_RATIO))
train_paths = paths[:split_idx]
val_paths = paths[split_idx:]

print(f'Total images: {len(paths)} | Train: {len(train_paths)} | Val: {len(val_paths)}')

Total images: 4322 | Train: 3673 | Val: 649


In [4]:
class RTTSFogPairs(Dataset):
    def __init__(self, image_paths: List[Path], img_size: int = 256, is_train: bool = False):
        self.image_paths = image_paths
        self.img_size = img_size
        self.is_train = is_train
        self.to_tensor = transforms.ToTensor()

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int):
        p = self.image_paths[idx]
        rgb = read_rgb(p)
        rgb = cv2.resize(rgb, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)

        # Data augmentation for training
        if self.is_train:
            # Random horizontal flip
            if np.random.rand() > 0.5:
                rgb = cv2.flip(rgb, 1)
            # Random rotation (small)
            if np.random.rand() > 0.6:
                angle = np.random.uniform(-10, 10)
                center = (self.img_size // 2, self.img_size // 2)
                M = cv2.getRotationMatrix2D(center, angle, 1.0)
                rgb = cv2.warpAffine(rgb, M, (self.img_size, self.img_size))
            # Random brightness
            if np.random.rand() > 0.5:
                brightness = np.random.uniform(0.9, 1.1)
                rgb = np.clip(rgb.astype(np.float32) * brightness, 0, 255).astype(np.uint8)
            # Random noise
            if np.random.rand() > 0.7:
                noise = np.random.normal(0, 5, rgb.shape)
                rgb = np.clip(rgb.astype(np.float32) + noise, 0, 255).astype(np.uint8)

        clean = pseudo_clean_from_fog(rgb)
        foggy, beta = synthesize_fog(clean, depth_variation=self.is_train)

        x = torch.from_numpy(foggy.transpose(2, 0, 1)).float()
        y = torch.from_numpy(clean.transpose(2, 0, 1)).float()
        fog_cls = beta_to_fog_class(beta)

        return x, y, torch.tensor(fog_cls, dtype=torch.long), torch.tensor(beta, dtype=torch.float32)

train_ds = RTTSFogPairs(train_paths, IMG_SIZE, is_train=True)
val_ds = RTTSFogPairs(val_paths, IMG_SIZE, is_train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

xb, yb, cb, bb = next(iter(train_loader))
print(xb.shape, yb.shape, cb.shape, bb.shape)

torch.Size([8, 3, 256, 256]) torch.Size([8, 3, 256, 256]) torch.Size([8]) torch.Size([8])


In [6]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, p=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, k, s, p),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, k, 1, p),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class FFAInspiredNet(nn.Module):
    # Enhanced FFA-inspired network with better classification head
    def __init__(self, base=48):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(3, base, 3, 1, 1),
            nn.BatchNorm2d(base),
            nn.ReLU(inplace=True),
        )
        
        # Deeper encoder blocks
        self.b1 = ConvBlock(base, base)
        self.b2 = ConvBlock(base, base * 2, 3, 2, 1)  # Stride 2 for downsampling
        self.b3 = ConvBlock(base * 2, base * 2)
        self.b4 = ConvBlock(base * 2, base, 3, 2, 1)  # Further downsample
        self.b5 = ConvBlock(base, base)
        
        # Up-sampling path with proper channel handling
        self.up1 = nn.UpsamplingNearest2d(scale_factor=2)
        self.b6 = ConvBlock(base * 3, base * 2)  # Fixed: expects base + base*2 = 144 channels
        self.up2 = nn.UpsamplingNearest2d(scale_factor=2)
        self.b7 = ConvBlock(base * 3, base)  # Fixed: expects base + base*2 = 144 channels

        # Channel attention
        self.chan_attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(base, base // 4, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base // 4, base, 1),
            nn.Sigmoid(),
        )
        
        self.tail = nn.Sequential(
            nn.Conv2d(base, base, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base, 3, 3, 1, 1),
        )

        # Improved fog severity classification head (without BatchNorm to avoid issues with small batch sizes)
        self.fog_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(4),
            nn.Flatten(),
            nn.Linear(base * 16, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, 3),
        )

    def forward(self, x):
        h0 = self.head(x)
        h1 = self.b1(h0) + h0
        
        h2 = self.b2(h1)
        h3 = self.b3(h2) + h2
        
        h4 = self.b4(h3)
        h5 = self.b5(h4) + h4
        
        # Decoder with skip connections
        h6 = self.up1(h5)  # 48 channels
        h6 = torch.cat([h6, h3], dim=1)  # Now 48 + 96 = 144 channels
        h6 = self.b6(h6)  # Output: 96 channels
        
        h7 = self.up2(h6)  # 96 channels
        h7 = torch.cat([h7, h1], dim=1)  # Now 96 + 48 = 144 channels
        h7 = self.b7(h7)  # Output: 48 channels

        # Attention
        a = self.chan_attn(h7)
        h7 = h7 * a

        out = torch.clamp(self.tail(h7) + x, 0.0, 1.0)
        fog_logits = self.fog_head(h5)  # Use deeper features for fog classification
        return out, fog_logits

def mse_metric(x, y):
    return float(F.mse_loss(x, y).item())

def psnr_metric(x, y, max_val=1.0):
    mse = F.mse_loss(x, y).item()
    if mse <= 1e-12:
        return 99.0
    return float(10.0 * math.log10((max_val ** 2) / mse))

def ssim_metric_batch(x, y, c1=0.01**2, c2=0.03**2):
    # Simplified global SSIM over batch (stable and fast)
    x_mu = x.mean(dim=[2, 3], keepdim=True)
    y_mu = y.mean(dim=[2, 3], keepdim=True)
    x_var = ((x - x_mu) ** 2).mean(dim=[2, 3], keepdim=True)
    y_var = ((y - y_mu) ** 2).mean(dim=[2, 3], keepdim=True)
    xy_cov = ((x - x_mu) * (y - y_mu)).mean(dim=[2, 3], keepdim=True)

    ssim_n = (2 * x_mu * y_mu + c1) * (2 * xy_cov + c2)
    ssim_d = (x_mu ** 2 + y_mu ** 2 + c1) * (x_var + y_var + c2)
    ssim = ssim_n / (ssim_d + 1e-8)
    return float(ssim.mean().item())

model = FFAInspiredNet(base=48).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
l1_loss = nn.L1Loss()
ce_loss = nn.CrossEntropyLoss(label_smoothing=0.1)  # Label smoothing for regularization

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

In [ ]:
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for x, y, fog_cls, _beta in pbar:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        fog_cls = fog_cls.to(DEVICE)

        pred, fog_logits = model(x)

        loss_dehaze = l1_loss(pred, y)
        loss_cls = ce_loss(fog_logits, fog_cls)
        # Gradually increase classification weight
        cls_weight = 0.1 + 0.3 * (epoch / EPOCHS)
        loss = loss_dehaze + cls_weight * loss_cls

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
        opt.step()

        train_loss += loss.item() * x.size(0)
        pbar.set_postfix(loss=float(loss.item()), lr=opt.param_groups[0]['lr'])

    train_loss /= max(len(train_ds), 1)
    scheduler.step()

    model.eval()
    val_mse, val_psnr, val_ssim = [], [], []
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y, fog_cls, _beta in val_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            fog_cls = fog_cls.to(DEVICE)

            pred, fog_logits = model(x)

            val_mse.append(mse_metric(pred, y))
            val_psnr.append(psnr_metric(pred, y))
            val_ssim.append(ssim_metric_batch(pred, y))

            cls_pred = torch.argmax(fog_logits, dim=1)
            y_true.extend(fog_cls.cpu().numpy().tolist())
            y_pred.extend(cls_pred.cpu().numpy().tolist())

    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )

    rec = {
        'epoch': epoch,
        'train_loss': float(train_loss),
        'val_mse': float(np.mean(val_mse)),
        'val_psnr': float(np.mean(val_psnr)),
        'val_ssim': float(np.mean(val_ssim)),
        'cls_accuracy': float(acc),
        'cls_precision_macro': float(pr),
        'cls_recall_macro': float(rc),
        'cls_f1_macro': float(f1),
    }
    history.append(rec)
    print(rec)

Epoch 1/15:  93%|█████████▎| 427/460 [11:12<00:10,  3.21it/s, loss=0.168, lr=0.0002]  

In [ ]:
# System metrics (latency, FPS)
model.eval()
warm = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad():
    for _ in range(10):
        _ = model(warm)

n_runs = 50
start = time.perf_counter()
with torch.no_grad():
    for _ in range(n_runs):
        _ = model(warm)
elapsed = time.perf_counter() - start
latency_ms = (elapsed / n_runs) * 1000.0
fps = n_runs / elapsed

print({'latency_ms': latency_ms, 'fps': fps})

## Optional XGBoost Integration (Useful Baseline + Fusion Feature Head)

This section reuses the tabular idea from `prepare_features.py` and `train_xgboost.py`, but applies it to **FFA dehazed outputs**.

Why useful:
- Gives an interpretable baseline from handcrafted fog features.
- Works as a fallback when deep model confidence is unstable.
- Can be fused later in your risk engine with neural outputs.

In [ ]:
xgb_metrics = {}

try:
    from xgboost import XGBClassifier
    from sklearn.model_selection import train_test_split

    def extract_tabular_features_rgb01(img_rgb01: np.ndarray) -> Dict[str, float]:
        img_u8 = np.clip(img_rgb01 * 255.0, 0, 255).astype(np.uint8)
        gray = cv2.cvtColor(img_u8, cv2.COLOR_RGB2GRAY)
        hsv = cv2.cvtColor(img_u8, cv2.COLOR_RGB2HSV)

        lap_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        grad_x = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
        grad_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
        grad_mag = cv2.magnitude(grad_x, grad_y)

        saturation = hsv[:, :, 1].astype(np.float32)
        value = hsv[:, :, 2].astype(np.float32)

        min_rgb = np.min(img_u8, axis=2)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
        dark = cv2.erode(min_rgb, kernel)

        color_att = value - saturation
        p05 = float(np.percentile(gray, 5))
        p95 = float(np.percentile(gray, 95))

        return {
            'gray_mean': float(gray.mean()),
            'gray_std': float(gray.std()),
            'gray_p10': float(np.percentile(gray, 10)),
            'gray_p90': float(np.percentile(gray, 90)),
            'contrast_p95_p05': p95 - p05,
            'laplacian_var': lap_var,
            'gradient_mean': float(grad_mag.mean()),
            'gradient_std': float(grad_mag.std()),
            'saturation_mean': float(saturation.mean()),
            'saturation_std': float(saturation.std()),
            'value_mean': float(value.mean()),
            'value_std': float(value.std()),
            'dark_channel_mean': float(dark.mean()),
            'dark_channel_std': float(dark.std()),
            'dark_channel_p10': float(np.percentile(dark, 10)),
            'dark_channel_p90': float(np.percentile(dark, 90)),
            'color_attenuation_mean': float(color_att.mean()),
            'color_attenuation_std': float(color_att.std()),
            'color_attenuation_p10': float(np.percentile(color_att, 10)),
            'color_attenuation_p90': float(np.percentile(color_att, 90)),
        }

    rows = []
    labels = []

    model.eval()
    with torch.no_grad():
        for x, _y, fog_cls, _beta in val_loader:
            x = x.to(DEVICE)
            pred, _fog_logits = model(x)
            pred_np = pred.detach().cpu().numpy().transpose(0, 2, 3, 1)
            cls_np = fog_cls.numpy()

            for i in range(pred_np.shape[0]):
                rows.append(extract_tabular_features_rgb01(pred_np[i]))
                labels.append(int(cls_np[i]))

    feat_df = pd.DataFrame(rows)
    y_arr = np.array(labels)

    if len(np.unique(y_arr)) >= 2 and len(feat_df) > 20:
        X_train, X_test, y_train, y_test = train_test_split(
            feat_df,
            y_arr,
            test_size=0.25,
            random_state=SEED,
            stratify=y_arr,
        )

        xgb = XGBClassifier(
            n_estimators=250,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=SEED,
            eval_metric='mlogloss',
        )
        xgb.fit(X_train, y_train)

        y_hat = xgb.predict(X_test)
        xgb_acc = accuracy_score(y_test, y_hat)
        xgb_pr, xgb_rc, xgb_f1, _ = precision_recall_fscore_support(
            y_test, y_hat, average='macro', zero_division=0
        )

        xgb_metrics = {
            'task': '3-class fog severity from dehazed-image handcrafted features',
            'accuracy': float(xgb_acc),
            'precision_macro': float(xgb_pr),
            'recall_macro': float(xgb_rc),
            'f1_macro': float(xgb_f1),
            'train_samples': int(len(X_train)),
            'test_samples': int(len(X_test)),
            'num_features': int(feat_df.shape[1]),
        }
        print('XGBoost metrics:', xgb_metrics)
    else:
        print('Skipping XGBoost stage: not enough samples/classes in validation split.')

except Exception as ex:
    print('XGBoost integration skipped:', ex)

In [ ]:
# Optional: Detection metrics from external YOLO predictions
# Expected file format for predictions JSON (list):
# [
#   {"image_id": "AM_Bing_211", "gt": [[x1,y1,x2,y2,cls], ...],
#    "pred": [[x1,y1,x2,y2,cls,score], ...]}
# ]

def iou_xyxy(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    union = area_a + area_b - inter + 1e-9
    return inter / union

def evaluate_detection_json(pred_json_path: Path, iou_thr=0.5):
    data = json.loads(pred_json_path.read_text(encoding='utf-8'))
    tps, fps_, fns = 0, 0, 0
    ious = []

    for item in data:
        gt = item['gt']
        pred = sorted(item['pred'], key=lambda x: x[5], reverse=True)
        used = set()

        for p in pred:
            best_iou = 0.0
            best_j = -1
            for j, g in enumerate(gt):
                if j in used or int(g[4]) != int(p[4]):
                    continue
                ov = iou_xyxy(p[:4], g[:4])
                if ov > best_iou:
                    best_iou = ov
                    best_j = j
            if best_iou >= iou_thr:
                tps += 1
                ious.append(best_iou)
                used.add(best_j)
            else:
                fps_ += 1

        fns += (len(gt) - len(used))

    precision = tps / (tps + fps_ + 1e-9)
    recall = tps / (tps + fns + 1e-9)
    ap50 = precision * recall
    mAP = ap50
    mean_iou = float(np.mean(ious)) if ious else 0.0

    return {
        'iou_mean': mean_iou,
        'precision_det': precision,
        'recall_det': recall,
        'AP50': ap50,
        'mAP': mAP,
    }

# Example usage:
# det_metrics = evaluate_detection_json(Path('models/yolo_predictions.json'))
# print(det_metrics)

In [ ]:
# Save model + consolidated metrics
final_epoch = history[-1] if history else {}
metrics = {
    'dehazing': {
        'MSE': final_epoch.get('val_mse'),
        'PSNR': final_epoch.get('val_psnr'),
        'SSIM': final_epoch.get('val_ssim'),
    },
    'classification': {
        'Accuracy': final_epoch.get('cls_accuracy'),
        'Precision_macro': final_epoch.get('cls_precision_macro'),
        'Recall_macro': final_epoch.get('cls_recall_macro'),
        'F1_macro': final_epoch.get('cls_f1_macro'),
    },
    'xgboost_tabular_head': xgb_metrics,
    'system': {
        'FPS': float(fps),
        'Latency_ms': float(latency_ms),
    },
    'history': history,
}

model_path = MODEL_DIR / 'ffa_rtts_dehaze_fog.pt'
metrics_path = MODEL_DIR / 'ffa_rtts_metrics.json'

torch.save({'model_state_dict': model.state_dict(), 'img_size': IMG_SIZE}, model_path)
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')

print('Saved model:', model_path)
print('Saved metrics:', metrics_path)
print(json.dumps(metrics['dehazing'], indent=2))
print(json.dumps(metrics['classification'], indent=2))
print(json.dumps(metrics['xgboost_tabular_head'], indent=2))
print(json.dumps(metrics['system'], indent=2))

In [ ]:
# Visualizations for training diagnostics and qualitative results
import matplotlib.pyplot as plt

if history:
    history_df = pd.DataFrame(history)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('FFA-Net RTTS Training Diagnostics', fontsize=16)

    axes[0, 0].plot(history_df['epoch'], history_df['train_loss'], marker='o', color='#1f77b4')
    axes[0, 0].set_title('Training Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(history_df['epoch'], history_df['val_psnr'], marker='o', color='#ff7f0e', label='PSNR')
    axes[0, 1].plot(history_df['epoch'], history_df['val_ssim'], marker='o', color='#2ca02c', label='SSIM')
    axes[0, 1].set_title('Dehazing Quality')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()

    axes[1, 0].plot(history_df['epoch'], history_df['cls_accuracy'], marker='o', color='#d62728', label='Accuracy')
    axes[1, 0].plot(history_df['epoch'], history_df['cls_f1_macro'], marker='o', color='#9467bd', label='F1 Macro')
    axes[1, 0].set_title('Fog Classification Metrics')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylim(0, 1.0)
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()

    axes[1, 1].plot(history_df['epoch'], history_df['val_mse'], marker='o', color='#8c564b')
    axes[1, 1].set_title('Validation MSE')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('MSE')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    final_summary = pd.DataFrame([
        {
            'Metric Group': 'Dehazing',
            'MSE': final_epoch.get('val_mse'),
            'PSNR': final_epoch.get('val_psnr'),
            'SSIM': final_epoch.get('val_ssim'),
            'Accuracy': None,
            'Precision': None,
            'Recall': None,
            'F1': None,
        },
        {
            'Metric Group': 'Classification',
            'MSE': None,
            'PSNR': None,
            'SSIM': None,
            'Accuracy': final_epoch.get('cls_accuracy'),
            'Precision': final_epoch.get('cls_precision_macro'),
            'Recall': final_epoch.get('cls_recall_macro'),
            'F1': final_epoch.get('cls_f1_macro'),
        },
    ])
    display(final_summary)

    model.eval()
    sample_batch = next(iter(val_loader))
    sample_x = sample_batch[0][:4].to(DEVICE)
    sample_y = sample_batch[1][:4].to(DEVICE)
    with torch.no_grad():
        sample_pred, sample_logits = model(sample_x)
        sample_prob = torch.softmax(sample_logits, dim=1)
        sample_class = torch.argmax(sample_prob, dim=1).cpu().numpy()

    sample_x_np = sample_x.cpu().numpy().transpose(0, 2, 3, 1)
    sample_y_np = sample_y.cpu().numpy().transpose(0, 2, 3, 1)
    sample_pred_np = sample_pred.cpu().numpy().transpose(0, 2, 3, 1)

    fig, axes = plt.subplots(len(sample_x_np), 3, figsize=(12, 4 * len(sample_x_np)))
    if len(sample_x_np) == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(len(sample_x_np)):
        axes[i, 0].imshow(sample_x_np[i])
        axes[i, 0].set_title('Input Foggy')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(sample_pred_np[i])
        axes[i, 1].set_title('FFA-Net Output')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(sample_y_np[i])
        axes[i, 2].set_title(f'Pseudo Clean Target | Fog Class: {sample_class[i]}')
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.show()

    if xgb_metrics:
        xgb_plot_df = pd.DataFrame([xgb_metrics])
        metric_cols = [c for c in ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro'] if c in xgb_plot_df.columns]
        if metric_cols:
            xgb_plot_df[metric_cols].T.plot(kind='bar', legend=False, figsize=(8, 4), color='#1f77b4')
            plt.title('XGBoost Tabular Head Metrics')
            plt.ylabel('Score')
            plt.ylim(0, 1.0)
            plt.grid(axis='y', alpha=0.3)
            plt.tight_layout()
            plt.show()
else:
    print('No training history available yet. Run the training cells first.')